# Scratch Studio Comments Explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/22552/kasotest/blob/main/ScratchCommentsExplorer.ipynb)

`db.zst.part0` と `db.zst.part1` を取得して結合し、Colab上で展開して Gradio UI を起動します。

- 全文検索 / LIKE検索
- ユーザー・期間・親/返信フィルタ
- **検索対象はDB全体、表示は1ページ200件**
- 投稿数ランキング / 日別推移
- 読み取り専用SQL

> 元のDBは `--long=31` で圧縮された zstd を2分割したものです。2ファイルを順番に連結してから、システムの `zstd --long=31` で展開します。


In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd
!pip -q install "gradio>=6.20,<7" pandas plotly requests


In [ ]:
from pathlib import Path
import shutil
import subprocess
import requests

PART_URLS = [
    "https://raw.githubusercontent.com/22552/kasotest/main/db.zst.part0",
    "https://raw.githubusercontent.com/22552/kasotest/main/db.zst.part1",
]
PART_PATHS = [Path("/content/db.zst.part0"), Path("/content/db.zst.part1")]
ZST_PATH = Path("/content/comments.sqlite.zst")
DB_PATH = Path("/content/comments.sqlite")

for url, path in zip(PART_URLS, PART_PATHS):
    tmp = Path(str(path) + ".tmp")
    tmp.unlink(missing_ok=True)
    print(f"[download] {path.name}")
    with requests.get(url, stream=True, timeout=(15, 180)) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length") or 0)
        done = 0
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(
                        f"\r  {done / 1024 / 1024:.1f}/{total / 1024 / 1024:.1f} MiB "
                        f"({done / total:.0%})",
                        end="",
                    )
    print()
    tmp.replace(path)

print("[join] joining db.zst.part0 + db.zst.part1...")
tmp_zst = Path(str(ZST_PATH) + ".tmp")
tmp_zst.unlink(missing_ok=True)
with tmp_zst.open("wb") as out:
    for path in PART_PATHS:
        with path.open("rb") as src:
            shutil.copyfileobj(src, out, length=4 * 1024 * 1024)
tmp_zst.replace(ZST_PATH)
print(f"[join] done: {ZST_PATH.stat().st_size / 1024 / 1024:.1f} MiB")

print("[zstd] decompressing with system zstd --long=31...")
tmp_db = Path(str(DB_PATH) + ".tmp")
DB_PATH.unlink(missing_ok=True)
tmp_db.unlink(missing_ok=True)
try:
    subprocess.run(
        ["zstd", "-d", "--long=31", "-f", str(ZST_PATH), "-o", str(tmp_db)],
        check=True,
    )
    tmp_db.replace(DB_PATH)
except Exception:
    tmp_db.unlink(missing_ok=True)
    raise

print(f"[ready] {DB_PATH} ({DB_PATH.stat().st_size / 1024 / 1024:.1f} MiB)")


In [ ]:
import math, sqlite3
import pandas as pd
import plotly.express as px
import gradio as gr

PAGE_SIZE = 200

def db_connect():
    con = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True, timeout=5, check_same_thread=False)
    con.execute('PRAGMA query_only=ON')
    con.execute('PRAGMA busy_timeout=5000')
    return con

def build_search(query, mode, user, start, end, kind):
    clauses, params = [], []

    if kind == '親コメント':
        clauses.append('c.is_reply=0')
    elif kind == '返信':
        clauses.append('c.is_reply=1')

    user = (user or '').strip()
    if user:
        clauses.append('c.user=? COLLATE NOCASE')
        params.append(user)

    start = (start or '').strip()
    if start:
        clauses.append('c.datetime>=?')
        params.append(start)

    end = (end or '').strip()
    if end:
        e = end + ('T23:59:59.999999Z' if len(end) == 10 else '')
        clauses.append('c.datetime<=?')
        params.append(e)

    q = (query or '').strip()

    if q and mode == 'FTS5':
        where = ' AND '.join(['comments_fts MATCH ?'] + clauses)
        base = f'''FROM comments_fts
                   JOIN comments c ON c.id=comments_fts.rowid
                   WHERE {where}'''
        select = '''SELECT c.id,c.parent_id,c.is_reply,c.user,c.user_id,
                           c.datetime,c.content,bm25(comments_fts) AS score '''
        order = ' ORDER BY score,c.datetime DESC'
        return select, base, order, [q] + params

    if q:
        clauses.insert(0, 'c.content LIKE ?')
        params.insert(0, f'%{q}%')
    where = (' WHERE ' + ' AND '.join(clauses)) if clauses else ''
    base = 'FROM comments c' + where
    select = '''SELECT c.id,c.parent_id,c.is_reply,c.user,c.user_id,
                       c.datetime,c.content '''
    order = ' ORDER BY c.datetime DESC'
    return select, base, order, params

def search_page(query, mode, user, start, end, kind, page):
    try:
        page = max(1, int(page or 1))
    except Exception:
        page = 1

    try:
        select, base, order, params = build_search(query, mode, user, start, end, kind)
        with db_connect() as con:
            total = con.execute('SELECT COUNT(*) ' + base, params).fetchone()[0]
            pages = max(1, math.ceil(total / PAGE_SIZE))
            page = min(page, pages)
            offset = (page - 1) * PAGE_SIZE
            df = pd.read_sql_query(
                select + base + order + ' LIMIT ? OFFSET ?',
                con,
                params=params + [PAGE_SIZE, offset]
            )
    except Exception as e:
        return pd.DataFrame(), f'❌ {type(e).__name__}: {e}', 1

    if not df.empty:
        df['種別'] = df['is_reply'].map({0:'親', 1:'返信'})
        cols = ['id','parent_id','種別','user','user_id','datetime','content']
        if 'score' in df.columns:
            cols.append('score')
        df = df[cols]

    first = 0 if total == 0 else offset + 1
    last = min(offset + len(df), total)
    status = f'✅ **{total:,}件ヒット** — {first:,}〜{last:,}件 / **{page:,}/{pages:,}ページ**'
    return df, status, page

def new_search(query, mode, user, start, end, kind):
    return search_page(query, mode, user, start, end, kind, 1)

def prev_page(query, mode, user, start, end, kind, page):
    return search_page(query, mode, user, start, end, kind, max(1, int(page or 1) - 1))

def next_page(query, mode, user, start, end, kind, page):
    return search_page(query, mode, user, start, end, kind, int(page or 1) + 1)

def jump_page(query, mode, user, start, end, kind, page):
    return search_page(query, mode, user, start, end, kind, page)

def stats():
    with db_connect() as con:
        total,top,replies,users,old,new = con.execute(
            '''SELECT COUNT(*),SUM(is_reply=0),SUM(is_reply=1),
                      COUNT(DISTINCT user),MIN(datetime),MAX(datetime)
               FROM comments'''
        ).fetchone()
    return f'''### DB概要
- 全行数: **{total:,}**
- 親コメント: **{top:,}**
- 返信: **{replies:,}**
- ユーザー数: **{users:,}**
- 期間: `{old}` ～ `{new}`'''

def top_users(n):
    with db_connect() as con:
        return pd.read_sql_query(
            'SELECT user,COUNT(*) comments FROM comments GROUP BY user ORDER BY comments DESC LIMIT ?',
            con, params=[int(n)]
        )

def daily_plot():
    with db_connect() as con:
        df = pd.read_sql_query(
            "SELECT substr(datetime,1,10) day,COUNT(*) comments FROM comments GROUP BY day ORDER BY day",
            con
        )
    return px.line(df, x='day', y='comments', title='日別コメント数')

def run_sql(sql):
    text = (sql or '').strip().rstrip(';').strip()
    if not text:
        return pd.DataFrame(), 'SQLを入力してください'
    if ';' in text:
        return pd.DataFrame(), '❌ 複数文は禁止'
    first = text.split(None,1)[0].upper()
    if first not in {'SELECT','WITH','EXPLAIN'}:
        return pd.DataFrame(), '❌ 読み取り専用です'
    try:
        with db_connect() as con:
            cur = con.execute(text)
            cols = [d[0] for d in (cur.description or [])]
            rows = cur.fetchmany(1001)
        clipped = len(rows) > 1000
        rows = rows[:1000]
        return pd.DataFrame(rows, columns=cols), f'✅ {len(rows)}行' + ('（先頭1000行）' if clipped else '')
    except Exception as e:
        return pd.DataFrame(), f'❌ {type(e).__name__}: {e}'

with gr.Blocks(title='Scratch Studio Comments Explorer') as demo:
    gr.Markdown('# 🔎 Scratch Studio Comments Explorer\nDB全体から検索し、結果を**1ページ200件**で表示します。DBは読み取り専用です。')

    with gr.Tab('検索'):
        with gr.Row():
            q = gr.Textbox(label='本文検索')
            mode = gr.Dropdown(['FTS5','部分一致（LIKE）'], value='FTS5', label='方式')
        with gr.Row():
            user = gr.Textbox(label='ユーザー')
            kind = gr.Dropdown(['すべて','親コメント','返信'], value='すべて', label='種別')
        with gr.Row():
            start = gr.Textbox(label='開始', placeholder='2023-09-05')
            end = gr.Textbox(label='終了')

        search_btn = gr.Button('検索', variant='primary')
        status = gr.Markdown()
        table = gr.Dataframe()

        with gr.Row():
            prev_btn = gr.Button('← 前の200件')
            page = gr.Number(value=1, minimum=1, precision=0, label='ページ')
            jump_btn = gr.Button('ページへ移動')
            next_btn = gr.Button('次の200件 →')

        inputs = [q, mode, user, start, end, kind]
        search_btn.click(new_search, inputs, [table, status, page])
        prev_btn.click(prev_page, inputs + [page], [table, status, page])
        next_btn.click(next_page, inputs + [page], [table, status, page])
        jump_btn.click(jump_page, inputs + [page], [table, status, page])

    with gr.Tab('統計'):
        s = gr.Markdown()
        sb = gr.Button('概要')
        sb.click(stats, outputs=s)
        n = gr.Slider(10,200,50,step=10,label='上位ユーザー数')
        ub = gr.Button('ランキング')
        ut = gr.Dataframe()
        ub.click(top_users, n, ut)
        pb = gr.Button('日別グラフ')
        plot = gr.Plot()
        pb.click(daily_plot, outputs=plot)

    with gr.Tab('SQL'):
        sql = gr.Code(
            value='SELECT user, COUNT(*) AS comments FROM comments GROUP BY user ORDER BY comments DESC LIMIT 100;',
            language='sql',
            label='SQL'
        )
        rb = gr.Button('実行', variant='primary')
        msg = gr.Markdown()
        out = gr.Dataframe()
        rb.click(run_sql, sql, [out, msg])

demo.queue(default_concurrency_limit=2).launch(share=True, debug=False)
